In [9]:
import fitz
import pytesseract
from PIL import Image
import cv2
import numpy as np
import re
import os

class PDFProcessor:
    def __init__(self):
        self.output_dir = "processed_pagesz"
        
    def process_pdf(self, pdf_path):
        """Process PDF automatically without user interaction"""
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
            
        doc = fitz.open(pdf_path)
        all_items = []
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            
            # Increase the resolution of the rendered page
            zoom = 4  # Increased zoom factor for better resolution
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            
            # Convert PyMuPDF pixmap to PIL Image
            img_data = pix.samples
            img = Image.frombytes("RGB", [pix.width, pix.height], img_data)
            
            print(f"\nProcessing page {page_num + 1}")
            
            # Enhanced preprocessing for better OCR
            processed_img = self.preprocess_image(img)
            
            # Save only the processed image
            processed_path = os.path.join(self.output_dir, f'page_{page_num + 1}.png')
            processed_img.save(processed_path, dpi=(300, 300))
            
            # Perform OCR with improved settings
            custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ/\"\\-. '
            text = pytesseract.image_to_string(processed_img, config=custom_config)
            
            # Extract and store BOM data
            items = self.extract_bom_data(text)
            all_items.extend(items)
        
        doc.close()
        return all_items
    
    @staticmethod
    def preprocess_image(image):
        """Enhanced preprocessing for better text clarity"""
        # Convert to OpenCV format
        opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        
        # Convert to grayscale
        gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
        
        # Apply adaptive thresholding
        binary = cv2.adaptiveThreshold(
            gray,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,  # Block size
            2    # C constant
        )
        
        # Denoise
        denoised = cv2.fastNlMeansDenoising(binary)
        
        # Optional: Apply slight sharpening
        kernel = np.array([[-1,-1,-1],
                         [-1, 9,-1],
                         [-1,-1,-1]])
        sharpened = cv2.filter2D(denoised, -1, kernel)
        
        return Image.fromarray(sharpened)
    
    @staticmethod
    def extract_bom_data(text):
        """Extract Bill of Materials data"""
        items = []
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        
        pattern = r'(?P<id>\d+)\s*(?P<qty>[\d\'-]+)\s*(?P<location>SHOP|FIELD)\s*(?P<nd>[\d/]+\"?)\s*(?P<description>.*)'
        
        for line in lines:
            match = re.search(pattern, line)
            if match:
                items.append({
                    'id': match.group('id'),
                    'qty': match.group('qty'),
                    'location': match.group('location'),
                    'nd': match.group('nd'),
                    'description': match.group('description').strip()
                })
        
        return items

def display_results(items):
    """Display the BOM results in a table format"""
    if not items:
        print("\nNo items were extracted from the PDF.")
        return
        
    print("\nBill of Materials:")
    print("=" * 100)
    print(f"{'ID':<5}{'QTY':<10}{'SHOP/FIELD':<12}{'ND':<8}{'DESCRIPTION':<65}")
    print("-" * 100)
    
    for item in items:
        print(f"{item['id']:<5}{item['qty']:<10}{item['location']:<12}{item['nd']:<8}{item['description']:<65}")

def main():
    print("PDF Bill of Materials Processor")
    print("==============================")
    pdf_path = input("Enter the path to your PDF file: ")
    
    if not os.path.exists(pdf_path):
        print(f"Error: File not found at {pdf_path}")
        return
        
    try:
        processor = PDFProcessor()
        items = processor.process_pdf(pdf_path)
        display_results(items)
        
        print(f"\nProcessed images have been saved to the '{processor.output_dir}' directory")
        
    except Exception as e:
        print(f"Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

PDF Bill of Materials Processor

Processing page 1

Processing page 2

Processing page 3

No items were extracted from the PDF.

Processed images have been saved to the 'processed_pagesz' directory


In [10]:
import fitz
import pytesseract
from PIL import Image
import cv2
import numpy as np
import re
import os

class PDFProcessor:
    def __init__(self):
        self.output_dir = "filters"
        
    def process_pdf(self, pdf_path):
        """Process PDF automatically without user interaction"""
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
            
        doc = fitz.open(pdf_path)
        all_items = []
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            
            # Increase the resolution of the rendered page
            zoom = 4  # Increased zoom factor for better resolution
            mat = fitz.Matrix(zoom, zoom)
            pix = page.get_pixmap(matrix=mat, alpha=False)
            
            # Convert PyMuPDF pixmap to PIL Image
            img_data = pix.samples
            img = Image.frombytes("RGB", [pix.width, pix.height], img_data)
            
            print(f"\nProcessing page {page_num + 1}")
            
            # Enhanced preprocessing for better OCR
            processed_img = self.preprocess_image(img)
            
            # Save only the processed image
            processed_path = os.path.join(self.output_dir, f'page_{page_num + 1}.png')
            processed_img.save(processed_path, dpi=(300, 300))
            
            # Perform OCR with improved settings
            custom_config = r'--oem 3 --psm 6 -c tessedit_char_whitelist=0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ/\"\\-. '
            text = pytesseract.image_to_string(processed_img, config=custom_config)
            
            # Extract and store BOM data
            items = self.extract_bom_data(text)
            all_items.extend(items)
        
        doc.close()
        return all_items
    
    @staticmethod
    def preprocess_image(image):
        """Enhanced preprocessing for better text clarity"""
        # Convert to OpenCV format
        opencv_img = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
        
        # Convert to grayscale
        gray = cv2.cvtColor(opencv_img, cv2.COLOR_BGR2GRAY)
        
        # Apply Gaussian blur to reduce noise
        blurred = cv2.GaussianBlur(gray, (3, 3), 0)
        
        # Apply adaptive thresholding for binarization
        binary = cv2.adaptiveThreshold(
            blurred,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            11,  # Block size
            2    # C constant
        )
        
        # Additional binary thresholding for cleaner results
        _, binary = cv2.threshold(binary, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Denoise
        denoised = cv2.fastNlMeansDenoising(binary)
        
        # Edge enhancement
        edges = cv2.Canny(denoised, 50, 150)
        dilated_edges = cv2.dilate(edges, None)
        
        # Combine the binary image with enhanced edges
        result = cv2.addWeighted(denoised, 0.8, dilated_edges, 0.2, 0)
        
        # Optional: Apply slight sharpening
        kernel = np.array([[-1,-1,-1],
                         [-1, 9,-1],
                         [-1,-1,-1]])
        sharpened = cv2.filter2D(result, -1, kernel)
        
        return Image.fromarray(sharpened)
    
    @staticmethod
    def extract_bom_data(text):
        """Extract Bill of Materials data"""
        items = []
        lines = [line.strip() for line in text.split('\n') if line.strip()]
        
        pattern = r'(?P<id>\d+)\s*(?P<qty>[\d\'-]+)\s*(?P<location>SHOP|FIELD)\s*(?P<nd>[\d/]+\"?)\s*(?P<description>.*)'
        
        for line in lines:
            match = re.search(pattern, line)
            if match:
                items.append({
                    'id': match.group('id'),
                    'qty': match.group('qty'),
                    'location': match.group('location'),
                    'nd': match.group('nd'),
                    'description': match.group('description').strip()
                })
        
        return items

def display_results(items):
    """Display the BOM results in a table format"""
    if not items:
        print("\nNo items were extracted from the PDF.")
        return
        
    print("\nBill of Materials:")
    print("=" * 100)
    print(f"{'ID':<5}{'QTY':<10}{'SHOP/FIELD':<12}{'ND':<8}{'DESCRIPTION':<65}")
    print("-" * 100)
    
    for item in items:
        print(f"{item['id']:<5}{item['qty']:<10}{item['location']:<12}{item['nd']:<8}{item['description']:<65}")

def main():
    print("PDF Bill of Materials Processor")
    print("==============================")
    pdf_path = input("Enter the path to your PDF file: ")
    
    if not os.path.exists(pdf_path):
        print(f"Error: File not found at {pdf_path}")
        return
        
    try:
        processor = PDFProcessor()
        items = processor.process_pdf(pdf_path)
        display_results(items)
        
        print(f"\nProcessed images have been saved to the '{processor.output_dir}' directory")
        
    except Exception as e:
        print(f"Error processing PDF: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

PDF Bill of Materials Processor

Processing page 1

Processing page 2

Processing page 3

No items were extracted from the PDF.

Processed images have been saved to the 'filters' directory
